# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [12]:
%pip -q install duckdb
import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")  # exact name you set in Colab Secrets
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [13]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [14]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_clients.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [15]:
scored = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    )
    SELECT r.content_hash_id, r.impressions, r.clicks,
           c.client_hash_id, c.content_type, c.search_volume
    FROM rollup r
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON r.content_hash_id = c.content_hash_id
    WHERE c.content_type = 'keyword article' AND c.search_volume >= 20
""").df()

print(f"Scored population: {len(scored)} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scored population: 38726 rows


In [16]:
n_clients = scored['client_hash_id'].nunique()
print(f"Distinct clients in scored population: {n_clients}")

Distinct clients in scored population: 43


In [17]:
pages_per_client = scored.groupby('client_hash_id').size()
print(pages_per_client.describe())
print()
print(pages_per_client.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

count       43.000000
mean       900.604651
std       2146.087316
min          1.000000
25%          6.500000
50%         64.000000
75%        703.500000
max      11127.000000
dtype: float64

0.10       2.20
0.25       6.50
0.50      64.00
0.75     703.50
0.90    2795.20
0.99    9462.12
dtype: float64


In [18]:
pages_per_client.sort_values(ascending=False).head(10)

,0
client_hash_id,
client_73cda7b4e4f265ea,11127
client_62f4a7e64f5e0096,7163
client_e547b89c05043229,4415
client_08a6a72ff48e62c0,4118
client_fef1a8f436438636,3156
client_2094c6eb080311d5,1352
client_e5c2aa26a8598242,1054
client_23a62021009f63c4,842
client_a80fca3f171ed1de,814


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [20]:
model_data = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
                WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
                WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
                ELSE 'tier_4_21plus'
            END AS position_tier
        FROM rollup
    ),
    benchmark(position_tier, expected_ctr) AS (
        VALUES
            ('tier_1_1-3', 0.002811), ('tier_2_4-10', 0.002353),
            ('tier_3_11-20', 0.002226), ('tier_4_21plus', 0.000748)
    ),
    labeled AS (
        SELECT
            t.content_hash_id, t.impressions, t.clicks, t.weighted_avg_position,
            b.expected_ctr,
            (b.expected_ctr * t.impressions) - t.clicks AS lost_clicks
        FROM tiered t
        JOIN benchmark b USING (position_tier)
    )
    SELECT
        l.content_hash_id, c.client_hash_id,
        l.lost_clicks, l.impressions, l.clicks, l.weighted_avg_position, l.expected_ctr,
        c.search_volume, c.competition_level, c.content_type,
        CASE WHEN c.content_updated_date <= DATE '2026-03-31'
             THEN DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')
             ELSE NULL END AS days_since_update_capped,
        c.word_count
    FROM labeled l
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON l.content_hash_id = c.content_hash_id
    WHERE c.content_type = 'keyword article' AND c.search_volume >= 20
""").df()

print(f"Rows: {len(model_data)}, distinct clients: {model_data['client_hash_id'].nunique()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 38726, distinct clients: 43


In [23]:
#feature preparation
model_data['has_word_count'] = model_data['word_count'].notna().astype(int)
model_data['word_count'] = model_data['word_count'].fillna(0)

model_data['updated_before_march'] = model_data['days_since_updated_capped'].notna().astype(int)
model_data['days_since_updated_capped'] = model_data['days_since_updated_capped'].fillna(0)

model_data = pd.get_dummies(model_data, columns=['competition_level'],drop_first = True)
# content_type is constant ('keyword article') in this scored slice, so it isn't a usable feature here
model_data['capture_ratio'] = model_data['clicks'] / model_data['search_volume'] # baseline rule's score — lower = higher priority

honest_features = ['search_volume', 'word_count', 'has_word_count',
                    'days_since_update_capped', 'updated_before_march31'] + \
                   [c for c in model_data.columns if c.startswith('competition_level_')]
print("Features used:", honest_features)

KeyError: 'days_since_updated_capped'

In [ ]:
def precision_at_k(true_ids_ranked, pred_ids_ranked, k=20):
  true_top_k = set(true_ids_ranked[:k])
  pred_top_k = set(pred_ids_ranked[:k])
  return len(true_top_k and pred_top_k) / k

def run_grouped_cv(data, feature_tools, k):
  gkf = GroupKFold(n_splits = k)
  groups = data['client_hash_id']
  X_all = data[feature_cols]
  y_all = data['lost_clicks']
  fold results = []

  for fold_i, (train_idx, test_idx) in enumerate(gkf.split(X_all, y_all, groups = groups)):
    train_df = data.iloc[train_idx]
    test_df = data.iloc[test_idx].copy()

    X_train, y_train = train_df[feature_cols], train_df['lost_clicks']
    X_test, y_test = test_df[feature_cols], test_df['lost_clicks']

    #ground truth ranking for this fold's held-out rows
    true_ranked_ids = test_df.sort_values('lost_clicks', ascending = False)['content_hash_id'].tolist()

    #baseline
    baseline_ranked_ids = test_df.sort_values('capture_ratio', ascending = True)['content_hash_id'].tolist()
    baseline_p20 = precision_at_k(true_ranked_ids, baseline_ranked_ids, k = 20)

    #linear regression
    lin = LinearRegression().fit(X_train, y_train)
    test_df['pred_lr'] = lin.predict(X_test)
    lr_ranked_ids = test_df.sort_values('pred_lr', ascending = False)['content_hash_id'].tolist()
    lr_p20 = precision_at_k(true_ranked_ids, lr_ranked_ids, k= 20)
    lr_r2 = r2_score(y_test, test_df['pred_lr'])

    #Random Forest
    rf = RandomForestRegressor(n_estimators = 300, random_state = 42, n_jobs = -1).fit(X_train, y_train)
    test_df['pred_df'] = rf.predict(X_test)
    rf_ranked_ids = test_df.sort_values('pred_rf', ascending = False)['content_hash_id'].to_list()
    rf_p20 = precision_at_k(true_ranked_ids, rf_ranked_ids, k = 20)
    rf_r2 = r2_score(y_test, test_df['pred_rf'])

    fold_results.append({
        'fold': fold_i,
        'n_test_clients': test_df['client_hash_id'].nunique(),
        'n_test_rows': len(test_df),
        'baseline_p20': baseline_p20,
        'lr_p20': lr_p20,
        'lr_r2': lr_r2,
        'rf_p20': rf_p20,
        'rf_r2': rf_r2
    })
  return pd.DataFrame(fold_results)


In [ ]:
results_k5 = run_grouped_cv(model_data, honest_features, k = 5)
results_k10 = run_grouped_cv(model_data, honest_features, k = 10)

print("k5 Folds")
print(results_k5)
print()
print("k10 Folds")
print(results_k10)


In [ ]:
summary = pd.DataFrame({
    'k=5': results_k5[['baseline_p20','lr_p20','lr_r2','rf_p20','rf_r2']].mean(),
    'k=10': results_k10[['baseline_20','lr_20','lr_r2','rf_20','rf_r2']].mean()
})

summary

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.